# 7. Latent Drifting and Learned Priors

Notebook 4 moved the *field* into a learned feature space while the generator kept producing pixels. This notebook
moves the **generator's output** as well, and then asks what happens when the space it outputs into is *itself being
learned*.

Three things get called "latent drifting" and they are not the same:

| | what is latent | who trains it | example |
|---|---|---|---|
| **(a) latent-space generation** | the generator's output lives in a frozen autoencoder's latent | pretrained, frozen | the paper's SD-VAE runs, FID 1.54 at 1 NFE |
| **(b) latent + non-autoregressive decode** | an $\ell_2$-normalised latent decoded to discrete tokens | trained jointly with the drift | Pepti-drift (Notebook 10) |
| **(c) learned prior** | the *prior* $p_{\boldsymbol\epsilon}$ is produced by another model | trained, possibly concurrently | Drift-AR |

(a) is an engineering convenience. (b) changes what the metric means. (c) introduces a genuinely new failure mode —
the **moving prior** — which is the thing you must understand before attempting to put a drifting module inside a
larger trainable system such as a structure-prediction network.

The claims we will establish:

1. Drifting in a frozen VAE latent works and is cheaper than pixel space, but **equilibrium in latent space is not
   equilibrium in pixel space** — the decoder's Jacobian reweights the metric, and we will measure how badly.
2. $\ell_2$-normalising the latent onto a sphere removes a real failure (radius drift) and makes $\tau$ a
   cosine-similarity threshold.
3. A prior that moves at the same rate the generator learns destroys training; the fix is **staging**, and there is a
   single diagnostic ratio that tells you whether you are in trouble.
4. Tying the prior's variance to a predictive model's per-position entropy makes the drift *automatically* vanish
   where the predictor is confident.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- the drifting kit from Notebook 3, unchanged ---------------------------------
def compute_V(x, y_pos, y_neg=None, tau=0.2):
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg: y_neg = x
    lp = -torch.cdist(x, y_pos) / tau
    ln = -torch.cdist(x, y_neg) / tau
    if self_neg: ln = ln - torch.eye(N, device=x.device) * 1e6
    logit = torch.cat([lp, ln], 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[0]; Ap, An = A[:, :P], A[:, P:]
    return (Ap * An.sum(1, keepdim=True)) @ y_pos - (An * Ap.sum(1, keepdim=True)) @ y_neg

def feature_scale(f, fp):
    with torch.no_grad():
        return (torch.cdist(f, fp).mean() / math.sqrt(f.shape[-1])).clamp_min(1e-8)

def drift_target(f, fp, taus=(0.05, 0.2, 0.8), eta=0.05):
    """Returns (loss, raw ||V||^2 at the finest tau, taus[0]) with everything normalised."""
    C = f.shape[-1]
    S = feature_scale(f, fp)
    fn, fpn = f / S, fp / S
    with torch.no_grad():
        V, raw = 0.0, None
        for t in taus:
            v = compute_V(fn, fpn, tau=t * math.sqrt(C))
            if raw is None: raw = v.pow(2).mean().item()
            V = V + v / v.pow(2).mean().sqrt().clamp_min(1e-8)
        target = fn + eta * V
    return F.mse_loss(fn, target), raw

## 1. A small VAE, then drift inside it

Nothing exotic: a convolutional VAE with a 16-dimensional latent, trained for a couple of minutes. The point is not
the VAE's quality; it is that the latent is **frozen** before drifting starts, so the target distribution
$p_{\text{data}}^{\text{latent}} = [\,e\,]_{\#}p_{\text{data}}$ is a fixed object.

In [ ]:
ds = torchvision.datasets.MNIST("./data", train=True, download=True)
X = ((ds.data.float() / 255.0) * 2 - 1).to(device)          # [60000, 28, 28] in [-1, 1]
Y = ds.targets.to(device)
print("data:", tuple(X.shape))

def real_batch(n):
    i = torch.randint(0, X.shape[0], (n,), device=device)
    return X[i].view(n, 1, 28, 28), Y[i]

D_Z = 16

class VAE(nn.Module):
    def __init__(self, d_z=D_Z, c=32):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, c, 4, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),        # 14
            nn.Conv2d(c, 2*c, 4, 2, 1), nn.GroupNorm(8, 2*c), nn.SiLU(),    # 7
            nn.Conv2d(2*c, 2*c, 3, 1, 1), nn.SiLU(), nn.Flatten(),
            nn.Linear(2*c*49, 2*d_z))
        self.dec = nn.Sequential(
            nn.Linear(d_z, 2*c*49), nn.SiLU(), nn.Unflatten(1, (2*c, 7, 7)),
            nn.ConvTranspose2d(2*c, c, 4, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.ConvTranspose2d(c, c, 4, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, 1, 3, 1, 1))
    def encode(self, x):
        mu, lv = self.enc(x).chunk(2, -1)
        return mu, lv.clamp(-8, 4)
    def decode(self, z): return torch.tanh(self.dec(z))

def train_vae(steps=3000, N=256, beta=1.0, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    v = VAE().to(device)
    opt = torch.optim.AdamW(v.parameters(), lr)
    for s in range(steps):
        x = real_batch(N)[0]
        mu, lv = v.encode(x)
        z = mu + torch.randn_like(mu) * (0.5 * lv).exp()
        rec = F.mse_loss(v.decode(z), x)
        kl = (-0.5 * (1 + lv - mu.pow(2) - lv.exp())).sum(-1).mean() / (28 * 28)
        loss = rec + beta * kl
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 750 == 0: print(f"  step {s:5d}  rec {rec.item():.4f}  kl {kl.item():.4f}")
    return v.eval().requires_grad_(False)

t0 = time.time(); vae = train_vae(); print(f"VAE trained in {time.time()-t0:.0f}s")

with torch.no_grad():
    xb = real_batch(8)[0]
    zb = vae.encode(xb)[0]
    print("latent per-dim std over the dataset:",
          vae.encode(real_batch(2048)[0])[0].std(0).cpu().numpy().round(2))
fig, ax = plt.subplots(2, 8, figsize=(11, 3))
for j in range(8):
    ax[0, j].imshow(xb[j, 0].cpu(), cmap="gray")
    ax[1, j].imshow(vae.decode(zb)[j, 0].cpu(), cmap="gray")
for a in ax.ravel(): a.axis("off")
ax[0, 0].set_title("input", fontsize=8); ax[1, 0].set_title("reconstruction", fontsize=8)
plt.tight_layout()

## 2. Drifting in the latent

The generator is now $f_\theta:\mathbb{R}^{d_z}\to\mathbb{R}^{d_z}$, noise to latent, and the field is computed on
raw latents — **no feature encoder**, because a VAE latent is already a space where Euclidean distance means
something (that is what the KL term buys you). Sampling is: draw $\boldsymbol\epsilon$, one MLP pass, one decoder
pass. Still 1 NFE for the generator.

The training loop is character-for-character the Notebook 3 loop with `X` replaced by `encode(X)`.

In [ ]:
@torch.no_grad()
def encode_batch(n):
    return vae.encode(real_batch(n)[0])[0]

class LatentGen(nn.Module):
    def __init__(self, d_z=D_Z, d_out=D_Z, h=512, nl=4):
        super().__init__()
        L, d = [], d_z
        for _ in range(nl): L += [nn.Linear(d, h), nn.SiLU()]; d = h
        self.net = nn.Sequential(*L, nn.Linear(d, d_out)); self.d_z = d_z
    def forward(self, z): return self.net(z)
    @torch.no_grad()
    def sample_latent(self, n): return self(torch.randn(n, self.d_z, device=device))

def train_latent_drift(steps=4000, N=256, P=256, lr=2e-4, eta=0.05,
                       taus=(0.05, 0.2, 0.8), sphere=False, seed=0, log_every=1000):
    torch.manual_seed(seed)
    g = LatentGen().to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=0.01)
    ema = {k: v.detach().clone() for k, v in g.state_dict().items()}
    for s in range(steps):
        zg = g(torch.randn(N, D_Z, device=device))
        zp = encode_batch(P)
        if sphere:
            zg, zp = F.normalize(zg, dim=-1), F.normalize(zp, dim=-1)
        loss, raw = drift_target(zg, zp, taus=taus, eta=eta)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
        d = min(0.999, (1 + s) / (10 + s))
        with torch.no_grad():
            for k, v in g.state_dict().items(): ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
        if s % log_every == 0: print(f"  step {s:5d}  loss {loss.item():.4f}  raw ||V||^2 {raw:.3e}")
    ge = LatentGen().to(device); ge.load_state_dict(ema)
    return ge

t0 = time.time(); g_lat = train_latent_drift(); print(f"latent drifting trained in {time.time()-t0:.0f}s")

@torch.no_grad()
def show(gen, sphere=False, n=10, title=""):
    z = gen.sample_latent(n)
    if sphere:
        z = F.normalize(z, dim=-1) * encode_batch(512).norm(dim=-1).mean()
    im = vae.decode(z)
    fig, ax = plt.subplots(1, n, figsize=(1.25 * n, 1.5))
    for j in range(n): ax[j].imshow(im[j, 0].cpu(), cmap="gray"); ax[j].axis("off")
    fig.suptitle(title, fontsize=9, y=1.06); plt.tight_layout()

show(g_lat, title="drifting in a frozen VAE latent, decoded (1 generator NFE)")

**Read the log before the pictures.** `raw ||V||^2` *rises* over training, from $4\times10^{-10}$ to $\sim10^{-2}$.
That is not divergence — it is Notebook 3 §8's **starved kernel**, seen from the inside. At initialisation the
generator emits nearly identical latents, so at the finest temperature ($\tau=0.05$) every pair is deep in the
exponential tail and the field is numerically zero. As the samples spread out, the fine scale starts to resolve
structure and its $\|\mathbf{V}\|$ becomes measurable. The coarse temperatures were doing all the work for the first
few hundred steps. This is exactly why the recipe sums over several $\tau$ instead of picking one, and why
$\|\mathbf{V}\|$ is logged per temperature rather than aggregated.

Two things to notice about the result. First, this is much cheaper than pixel-space drifting: the field is a
$256\times256$ distance matrix in 16 dimensions instead of 784, and the generator is a small MLP. Second — and this
is the actual argument for latent drifting — the metric problem of Notebook 4 mostly disappears. The VAE's KL term
has already forced the latent to be roughly isotropic and roughly Gaussian (check the per-dimension standard
deviations printed above: all near 1), so Euclidean distance is *approximately* perceptual, which is the property the
kernel needs. The paper's ImageNet results use precisely this: SD-VAE latents.

Let us check the "approximately" quantitatively, because it is where the argument leaks.

## 3. Equilibrium in latent space is not equilibrium in pixel space

Drifting drives $q_z\to p_z$. What we want is $q_x\to p_x$. These coincide only if the decoder $D$ is injective and
the pushforward commutes with the matching — and even then, the *rate* at which the field corrects errors is warped
by $\|\partial D/\partial z\|$, because a latent displacement $\delta$ produces a pixel displacement
$J\delta$ with $J$ varying by orders of magnitude across the latent space.

Concretely: two latents at the same distance can decode to images at wildly different distances. Where $J$ is small,
the field thinks it has separated two samples when the decoder has not.

In [ ]:
with torch.no_grad():
    z = encode_batch(512)
    x = vae.decode(z).view(512, -1)
    dz = torch.cdist(z, z); dx = torch.cdist(x, x)
    iu = torch.triu_indices(512, 512, offset=1)
    dzf, dxf = dz[iu[0], iu[1]], dx[iu[0], iu[1]]
    r = torch.corrcoef(torch.stack([dzf, dxf]))[0, 1].item()
    ratio = (dxf / dzf.clamp_min(1e-6))
    print(f"corr(latent distance, pixel distance) = {r:.3f}")
    print(f"pixel/latent distance ratio: median {ratio.median():.3f}, "
          f"5th pct {ratio.quantile(0.05):.3f}, 95th pct {ratio.quantile(0.95):.3f}, "
          f"spread {ratio.quantile(0.95)/ratio.quantile(0.05):.1f}x")

    # local Jacobian scale, sampled by finite differences
    zc = encode_batch(64)
    eps = 1e-2
    js = []
    for _ in range(8):
        d_ = F.normalize(torch.randn_like(zc), dim=-1) * eps
        js.append(((vae.decode(zc + d_) - vae.decode(zc)).view(64, -1).norm(dim=-1) / eps))
    J = torch.stack(js)
    print(f"local |J| across latent space: min {J.min():.2f}  median {J.median():.2f}  max {J.max():.2f}"
          f"   ({J.max()/J.min():.1f}x)")

A correlation of roughly 0.65–0.75 between latent and pixel distance, and a local Jacobian that varies about
sixfold across the latent, is the normal situation. It means three practical things:

1. **The kernel temperature is a latent-space quantity.** A $\tau$ that means "5% of typical distance" in latents
   means different perceptual things in different regions — a real bias that no tuning removes. But it is *bounded*,
   which raw pixels are not (Notebook 4 §1: pixel distance is better than chance but far from a semantic metric). A correlation of 0.67
   is a poor metric and an enormous improvement at the same time.
2. **A tighter VAE is a better drifting space, up to a point.** Raise $\beta$ and the latent becomes more isotropic
   (better metric) but the decoder loses detail (worse ceiling). The trade-off has an interior optimum. Worth
   re-running `train_vae(beta=...)` and re-measuring the correlation to see the first half of that curve.
3. **Never evaluate in latent space.** The drifting loss can fall while decoded samples get worse. Log a pixel-space
   or perceptual metric separately, exactly as Notebook 3 §7 insists on logging raw $\|\mathbf{V}\|^2$ rather than the
   normalised loss.

The same warning applies with much more force in structure prediction, where "latent" might be a frame or token
representation and the decoder is a geometric reconstruction whose Jacobian is genuinely singular in places.

## 4. Putting the latent on a sphere

Pepti-drift (`THEORY.md` §13.1) $\ell_2$-normalises latents, $u=\mathrm{norm}(\phi(e(y)))\in\mathbb{S}^{d_z-1}$.
Two reasons, one obvious and one not:

* **Obvious:** on the sphere, $\|u-v\|^2 = 2-2\langle u,v\rangle$, so the Laplace/Gaussian kernel becomes a monotone
  function of cosine similarity. $\tau$ becomes a *similarity threshold*, dimensionless and transferable — the same
  normalisation goal as Notebook 3 §1, achieved by construction instead of by an estimated scale $S$.
* **Non-obvious:** it removes a degree of freedom that drifting cannot control well. The radial direction is the one
  along which attraction and repulsion are most nearly parallel, so the equilibrium radius is set by a delicate
  cancellation. Left free, latents creep toward the origin (where the repulsive field from *all* other samples is
  most nearly balanced) and the decoder receives out-of-distribution inputs.

In [ ]:
with torch.no_grad():
    Z_data = encode_batch(4096)
    r_data = Z_data.norm(dim=-1).mean().item()

def latent_trace(sphere, steps=2500, seed=0):
    """Track (a) the radius of the latent that actually reaches the decoder and
    (b) how far generated latents sit from the nearest real latent."""
    torch.manual_seed(seed)
    g = LatentGen().to(device)
    opt = torch.optim.AdamW(g.parameters(), 2e-4, betas=(0.9, 0.95))
    rad, nnd, step = [], [], []
    for s in range(steps):
        zg = g(torch.randn(256, D_Z, device=device))
        zp = encode_batch(256)
        a, b = (F.normalize(zg, dim=-1), F.normalize(zp, dim=-1)) if sphere else (zg, zp)
        loss, _ = drift_target(a, b, eta=0.05)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
        if s % 50 == 0:
            with torch.no_grad():
                z_dec = F.normalize(zg, dim=-1) * r_data if sphere else zg   # what the decoder sees
                rad.append(z_dec.norm(dim=-1).mean().item())
                nnd.append(torch.cdist(z_dec, Z_data).min(1).values.mean().item())
                step.append(s)
    return step, rad, nnd

st, rad_f, nn_f = latent_trace(False)
_,  rad_s, nn_s = latent_trace(True)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].plot(st, rad_f, label="free latent")
ax[0].plot(st, rad_s, label="l2-normalised (radius supplied at decode)")
ax[0].axhline(r_data, ls="--", c="k", lw=1, label="data latent radius")
ax[0].set_xlabel("step"); ax[0].set_ylabel("mean ||z|| reaching the decoder"); ax[0].legend(fontsize=7)
ax[1].plot(st, nn_f, label="free latent")
ax[1].plot(st, nn_s, label="l2-normalised")
ax[1].set_xlabel("step"); ax[1].set_ylabel("distance to nearest real latent"); ax[1].legend(fontsize=7)
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout()
print(f"final radius reaching the decoder:  free {rad_f[-1]:.2f}   sphere {rad_s[-1]:.2f}   data {r_data:.2f}")
print(f"free-latent radius error:           {abs(rad_f[-1]-r_data)/r_data:.1%}")
print(f"final distance to nearest real latent: free {nn_f[-1]:.3f}   sphere {nn_s[-1]:.3f}")

With the free latent, the radius is a quantity the field must get right *by cancellation* between attraction and
repulsion — and it settles ~9% below the data's and keeps wandering. The decoder therefore receives inputs
systematically closer to the origin than real encodings, which is where a VAE decoder produces its blandest outputs.
With normalisation the field never sees the radius: you supply it at decode time (here, the data's mean radius;
Pepti-drift's decoder is trained on normalised latents so it never needs one) and what the field optimises is purely
angular.

The right-hand panel is the honest counterweight, and it is the reason to run this experiment rather than assume the
answer. On *distance to the nearest real latent* the spherical run is slightly **worse** here. Normalising fixed the
radius by fiat, but it also deleted a degree of freedom the field was using: with 16 dimensions and a VAE latent
whose radius genuinely carries information ("how typical is this sample"), projecting onto the sphere throws that
information away, and the angular problem that remains is no easier.

So: normalise when the latent is *yours to design* — as in Notebook 10, where the encoder emits unit-norm latents and the
decoder is trained on them, so it never sees a radius — and not when you are borrowing someone's pretrained VAE, where the
radius means something. This is a design decision with a measurable cost on both sides, not a free improvement.

## 5. The moving prior

Now the case that matters for putting drifting inside a bigger system. Suppose the prior is not $\mathcal{N}(0,I)$
but the output of another module $A_\psi$, and $\psi$ is being trained at the same time:

$$\boldsymbol\epsilon \sim \mathcal{N}\big(m_\psi(c),\ \sigma^2 I\big), \qquad
x = f_\theta(\boldsymbol\epsilon, c).$$

The drifting field measures the discrepancy between $q_\theta = [f_\theta]_\#p_{\boldsymbol\epsilon}^{\psi}$ and
$p_{\text{data}}$. If $\psi$ moves, $q_\theta$ moves **even when $\theta$ is frozen**. The generator is regressing
onto a target computed from samples of a distribution that will not be the same next step. Drift-AR names this the
*moving prior* problem (`THEORY.md` §13.2).

There is a clean way to think about whether you are in trouble. Per step, the drifting update moves the model
distribution by roughly $\eta\,\|\mathbf{V}\|\times(\text{lr fraction})$; the prior update moves it by
$\|\Delta m_\psi\|$. Define

$$\rho \;=\; \frac{\|\Delta m_\psi\|\ \text{per step}}{\eta\,\|\mathbf{V}\|\ \text{per step}}.$$

$\rho \ll 1$: the prior is quasi-static and drifting sees an effectively fixed problem. $\rho \gtrsim 1$: the target
moves as fast as you can chase it. Let us measure $\rho$ and the resulting damage on a conditional 2-D task.

In [ ]:
NC = 4
ANG = torch.arange(NC) * 2 * math.pi / NC
CENT = torch.stack([ANG.cos(), ANG.sin()], -1).to(device)

def cond_data(n):
    c = torch.randint(0, NC, (n,), device=device)
    t = torch.rand(n, device=device) * 2.2 - 1.1
    local = torch.stack([0.32 * t, 0.22 * t.pow(2) - 0.06], -1)      # a crescent per class
    ang = ANG.to(device)[c]
    R = torch.stack([torch.stack([ang.cos(), -ang.sin()], -1),
                     torch.stack([ang.sin(),  ang.cos()], -1)], -2)
    return CENT[c] + (R @ local.unsqueeze(-1)).squeeze(-1) + 0.03 * torch.randn(n, 2, device=device), c

def tv2d(A, B, bins=40, lim=1.6):
    def h(z):
        H = torch.histogramdd(z.cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0]
        return H / H.sum()
    return 0.5 * (h(A) - h(B)).abs().sum().item()

class PriorModule(nn.Module):
    """Stands in for Drift-AR's autoregressive backbone: predicts a per-class prior mean."""
    def __init__(self):
        super().__init__(); self.emb = nn.Embedding(NC, 2)
        nn.init.normal_(self.emb.weight, std=1.0)
    def forward(self, c): return self.emb(c)

class CondGen(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        self.emb = nn.Embedding(NC, 32)
        self.net = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, eps, c): return self.net(torch.cat([eps, self.emb(c)], -1))

def run(lr_psi, prior_pretrain=0, drift_steps=3000, N=256, eta=0.05, seed=0):
    """lr_psi = 0         -> the prior is frozen at initialisation
    lr_psi > 0         -> the prior co-trains with the drift, at that learning rate
    prior_pretrain = n -> train the prior ALONE for n steps first, then freeze it (staging)"""
    torch.manual_seed(seed)
    A, g = PriorModule().to(device), CondGen().to(device)
    oA = torch.optim.Adam(A.parameters(), max(lr_psi, 1e-2))
    og = torch.optim.Adam(g.parameters(), 2e-3, betas=(0.9, 0.95))
    for _ in range(prior_pretrain):                       # stage 1: prior only, no drifting
        y, c = cond_data(N)
        lA = F.mse_loss(A(c), y.detach())
        oA.zero_grad(); lA.backward(); oA.step()
    for gp in oA.param_groups: gp["lr"] = lr_psi
    hist = {"tv": [], "rho": [], "step": []}
    for s in range(drift_steps):                          # stage 2: drifting
        y, c = cond_data(N)
        m_before = A.emb.weight.detach().clone()
        if lr_psi > 0:
            lA = F.mse_loss(A(c), y.detach())
            oA.zero_grad(); lA.backward(); oA.step()
        dm = (A.emb.weight.detach() - m_before).norm(dim=-1).mean().item()
        eps = A(c).detach() + 0.35 * torch.randn(N, 2, device=device)
        x = g(eps, c)
        loss, Vmag = 0.0, 0.0
        for k in range(NC):
            mk = c == k
            if mk.sum() < 8: continue
            l, raw = drift_target(x[mk], y[mk], eta=eta)
            loss = loss + l * mk.float().mean(); Vmag += math.sqrt(raw)
        og.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); og.step()
        if s % 100 == 0:
            with torch.no_grad():
                yv, cv = cond_data(6000)
                ev = A(cv).detach() + 0.35 * torch.randn(6000, 2, device=device)
                hist["tv"].append(tv2d(g(ev, cv), yv))
                hist["rho"].append(dm / max(eta * Vmag / NC, 1e-9))
                hist["step"].append(s)
    return hist

runs = {
    "frozen prior (never trained)":        run(lr_psi=0.0),
    "co-trained, slow (lr 1e-3)":          run(lr_psi=1e-3),
    "co-trained, fast (lr 5e-2)":          run(lr_psi=5e-2),
    "staged: 800 prior-only, then frozen": run(lr_psi=0.0, prior_pretrain=800),
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, h in runs.items():
    ax[0].plot(h["step"], h["tv"], label=k)
    ax[1].semilogy(h["step"], np.maximum(h["rho"], 1e-6), label=k)
ax[0].set_xlabel("drifting step"); ax[0].set_ylabel("TV to data"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=7)
ax[1].axhline(1.0, c="k", ls="--", lw=1)
ax[1].set_xlabel("drifting step"); ax[1].set_ylabel(r"$\rho$ = prior motion / drift motion")
ax[1].grid(alpha=0.3); plt.tight_layout()

print(f"{'run':38s} {'final TV':>9s} {'best TV':>9s} {'median rho':>11s}")
for k, h in runs.items():
    print(f"{k:38s} {h['tv'][-1]:9.3f} {min(h['tv']):9.3f} {np.median(h['rho']):11.2e}")

Read the two panels together, because the right one explains the left one.

* **Frozen prior** ($\rho=0$) is the control: a fixed, arbitrary per-class mean, with the drift doing all the work.
* **Slow co-training** ($\rho\approx0.2$) matches it. The prior is quasi-static on the timescale of the drift, so
  drifting sees a slowly-varying problem and tracks it without difficulty. This is the regime you want if you must
  co-train.
* **Fast co-training** ($\rho\approx4$) is clearly worse and its TV curve is non-monotone: the generator repeatedly
  learns to match a prior that has moved by the time it gets there.
* **Staging** — train the prior first, freeze it, *then* drift — matches the frozen control while starting from a
  better place, because the prior is now informative rather than arbitrary. This is Drift-AR's prescription and it is
  the right default.

$\rho$ is the number to watch, and it costs nothing: you already log $\|\mathbf{V}\|$ (Notebook 3 §7), and
$\|\Delta m_\psi\|$ is one line. The threshold is soft but the shape is clear — below $\sim10^{-1}$ you are fine,
around 1 you are chasing.

**Why this matters beyond Drift-AR.** Any plan to train a drifting head *inside* a larger network — a
structure-prediction trunk, a policy encoder, a jointly-learned feature space — has a moving prior, and often a
moving *metric* as well. The metric case is strictly worse: if the feature encoder $\phi_\psi$ is trainable, the
target $p$ moves in feature space too, so the field's own definition changes underneath the regression and there is
no fixed point to converge to at all. The remedies are the same in every case: freeze, stage, or slow with an EMA,
and log $\rho$.

## 6. Entropy-scaled priors: making the drift vanish where you are already right

Drift-AR's other idea is worth stealing on its own. Rather than a fixed prior width, tie the per-position variance to
the predictive model's uncertainty (`THEORY.md` eq. 61):

$$\sigma(\mathcal{E}) = \sigma_{\max}\,\frac{e^{\mathcal{E}/\tau_\sigma}-1}{e^{\mathcal{E}_{\max}/\tau_\sigma}-1}.$$

At a position the backbone is confident about, $\sigma\to0$, the prior is a point mass at the backbone's prediction,
and the generator has nothing to do — the drift there is essentially zero *by construction*, not by learning. At an
uncertain position the prior is broad and the field does the work. Generative capacity is spent exactly where the
deterministic predictor fails.

The toy: 8 positions, each a 2-D embedding. Positions 0–3 are deterministic given the input; positions 4–7 are
genuinely bimodal. A backbone predicts a mean and an entropy per position. We check that the drift concentrates on
the uncertain half.

In [ ]:
L, DPOS = 8, 2
def seq_data(n):
    """Positions 0..3 deterministic; 4..7 bimodal (the generative work is all in the second half)."""
    base = torch.tensor([[0.9, 0.1], [-0.7, 0.4], [0.2, -0.8], [-0.3, -0.5]], device=device)
    out = torch.zeros(n, L, DPOS, device=device)
    out[:, :4] = base + 0.03 * torch.randn(n, 4, DPOS, device=device)
    sign = (torch.randint(0, 2, (n, 4, 1), device=device).float() * 2 - 1)
    out[:, 4:] = sign * torch.tensor([[0.8, 0.6]], device=device) + 0.06 * torch.randn(n, 4, DPOS, device=device)
    return out

def sigma_of_entropy(E, sig_max=1.0, E_max=math.log(2), tau_sigma=0.3):
    return sig_max * (torch.expm1(E / tau_sigma)) / math.expm1(E_max / tau_sigma)

with torch.no_grad():
    Y8 = seq_data(8192)
    mean_pred = Y8.mean(0)                                  # the backbone's point prediction
    # a two-mode predictive entropy estimate per position, from the empirical spread
    spread = Y8.std(0).mean(-1)
    E = torch.where(spread > 0.3, torch.full_like(spread, math.log(2)), torch.zeros_like(spread))
    sig = sigma_of_entropy(E)
print("position:        ", list(range(L)))
print("spread:          ", spread.cpu().numpy().round(3))
print("entropy E:       ", E.cpu().numpy().round(3))
print("prior sigma(E):  ", sig.cpu().numpy().round(3))

# the drift a fresh generator feels, per position, under the entropy-scaled prior
torch.manual_seed(0)
eps = mean_pred + sig[:, None] * torch.randn(256, L, DPOS, device=device)
x = eps.reshape(256, -1)
y = seq_data(256).reshape(256, -1)
S = feature_scale(x, y)
V = compute_V(x / S, y / S, tau=0.2 * math.sqrt(x.shape[-1]))
per_pos = V.view(256, L, DPOS).norm(dim=-1).mean(0)
print("\n||V|| per position:", per_pos.cpu().numpy().round(3))
print(f"mean ||V|| over confident positions 0-3: {per_pos[:4].mean():.3f}")
print(f"mean ||V|| over uncertain positions 4-7: {per_pos[4:].mean():.3f}"
      f"   ({per_pos[4:].mean()/per_pos[:4].mean():.1f}x)")

The field concentrates on the positions the backbone could not resolve, and it does so without any per-position
weighting in the loss — the prior's own variance carries the information. This is a small idea with a large
consequence for scientific applications: it lets you keep a strong deterministic predictor and add generative
capacity **only** where the prediction is genuinely multi-modal, rather than replacing the predictor with a
generative model everywhere.

It is also the cleanest available answer to "how do I add sampling diversity to a model that is already accurate?" —
which is precisely the Protenix question (`PROTENIX_DRIFTING.md`). Note the prerequisite: you need a calibrated
uncertainty. An overconfident backbone will zero out the prior at positions that genuinely needed sampling, and
drifting cannot recover what the prior never proposed.

## Summary

* **Latent drifting** is the same algorithm with `encode(X)` in place of `X`. It is cheaper, and a KL-regularised
  latent is already an adequate metric space, which is why the paper's best image results are latent-space results.
* But **equilibrium in latent space is not equilibrium in data space**: the decoder Jacobian varies severalfold
  across the latent, so $\tau$ means different things in different regions. Never evaluate in latent space.
* **$\ell_2$-normalising** turns $\tau$ into a cosine threshold and removes the radial degree of freedom the field
  controls worst. Do it when the latent is yours to design.
* **The moving prior** is the failure mode to understand before embedding drifting in a larger trainable system.
  Diagnose with $\rho = \|\Delta m_\psi\| / (\eta\|\mathbf{V}\|)$; fix by staging or slowing. If the *metric* is also
  being learned, the problem is strictly worse.
* **Entropy-scaled priors** put generative capacity exactly where a deterministic predictor is uncertain, and make
  the drift vanish elsewhere by construction.

Next: [8. Equivariant Drifting on SE(3)](<8. Equivariant Drifting on SE(3).ipynb>) — the geometry you need before any
of this touches a molecule.